In [ ]:
#[-FINAL SCRIPT-]
#------------python lib--------------#
!pip install torch transformers tiktoken datasets
#------------------------------------#

import os
import torch
import tiktoken
from google.colab import drive
# remeber to Mount Drive here
drive.mount ('/content/drive')
# try to Setup Device & Tokenizer here
device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = tiktoken.get_encoding ("gpt2")
checkpoint_dir = '/content/drive/My Drive/AvasGPT_Checkpoints'
print(f"Environment Ready! Running on:{device}")
print(f"Tokenizer 'enc' is defined.")

import torch
import torch.nn as nn
from torch.nn import functional as F

# --- NEW ATTENTION HEAD BLOCKS (Building the manual brains) ---

class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    """ The non-linear computation part """
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication (SA) followed by computation (FFWD) """
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# --- UPDATED SIMPLEGPT ---

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.2):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.dropout = nn.Dropout(dropout)

        # Swapping EncoderLayer for our custom decoder Blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Token and position embeddings here
        tok_emb = self.token_embedding_table(idx) # (B,T,C) here
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) here
        x = self.dropout(tok_emb + pos_emb)

        # Passing through the manual Multi-Head blocks
        x = self.blocks(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# Initialize finally working
model = SimpleGPT(vocab_size=enc.n_vocab).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
params = sum(p.numel() for p in model.parameters())/1e6
print(f"AvasGPT initialized with {params:.2f}M parameters.")

# 3. test generate text
def generate_text(model, prompt, max_new_tokens=100, temperature=0.8, top_k=50):
    model.eval()
    idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # Using block_size from the model attributes
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        # TOP-K SAMPLING: Filter out the "junk" words
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    model.train()
    return enc.decode(idx[0].tolist())

# tokenization
from datasets import load_dataset
import datasets
import tiktoken
import numpy as np

#Load the dataset in streaming mode and this will help to partially download it hopefully
ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
# we can also try ScaleAI's Atlas QnA dataset after it lerns enough vocab

# 2. Remeber to Setup the Tokenizer (use the GPT-2 encoding cuz gpt-2 taste better than key)
enc = tiktoken.get_encoding("gpt2")
# its actually impossible for me to make an encoding from scratch

def get_batch(dataset_iterator, batch_size, block_size):
    """
    try to grab a chunk of text and format it for the model.
    """
    texts = [next(dataset_iterator)['text'] for _ in range(batch_size)]
    all_tokens = []
    for t in texts:
        all_tokens.extend(enc.encode_ordinary(t))

    if len(all_tokens) < (batch_size * block_size + 1):
        # [][][][][][][][][][][][][[][][][][][][][]][][][][]
        return get_batch(dataset_iterator, batch_size, block_size)

    data = torch.tensor(all_tokens[:batch_size * block_size + 1], dtype=torch.long)
    x = data[:-1].view(batch_size, block_size)
    y = data[1:].view(batch_size, block_size)

    return x.to(device), y.to(device)

# Remeber to add Initialization for the stream
data_iter = iter(ds)

# Test it rn
X, Y = get_batch(data_iter, batch_size=4, block_size=8)
print("Data looks good! Shape of input:", X.shape)
print("Sample tokens for AvasGPT to learn:", X[0])

import os
import math

# --- Configs ---
max_iters = 60000
eval_interval = 500
batch_size = 32
block_size = 256
learning_rate = 3e-4
checkpoint_path = os.path.join(checkpoint_dir, 'avas_latest.pt')

def get_lr(it):
  #----Psuedocode comment----#
    # 1) Linear warmup for 500 steps
    if it < 500: return learning_rate * it / 500
    # 2) If it > max_iters, return min learning rate
    if it > max_iters: return learning_rate * 0.1
    # 3) In between, use cosine decay
    decay_ratio = (it - 500) / (max_iters - 500)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return 0.1 * learning_rate + coeff * learning_rate

start_iter = 0
if os.path.exists(checkpoint_path):
    print(f"Resuming AvasGPT...")
    checkpoint = torch.load(checkpoint_path)
    # Note: If weights don't match your new architecture, delete the old file!
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_iter = checkpoint['step']

print("Training with Cosine Decay...")

for i in range(start_iter, max_iters):
    lr = get_lr(i)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    x, y = get_batch(data_iter, batch_size, block_size)
    logits, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

# add training loop here
    if i % eval_interval == 0:
        print(f"Step {i} | Loss {loss.item():.4f} | LR {lr:.2e}")
        torch.save({'step': i, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, checkpoint_path)
        print(f"Sample: {generate_text(model, 'Artificial Intelligence is', max_new_tokens=40)}")

# -------------------------------------------------------------------------------#
